In [10]:
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pandas as pd
import os
from urllib.parse import urlparse


file_path = "data/fineweb/004_00048.parquet"


# ===== 1. 文件与 parquet 元信息 =====
pf = pq.ParquetFile(file_path)

print("=== 基本信息 Basic Info ===")
print("File path:", os.path.abspath(file_path))
print(f"File size: {os.path.getsize(file_path) / (1024 ** 2):.2f} MB")
print("Row groups:", pf.num_row_groups)
print("Number of rows:", pf.metadata.num_rows)
print("Number of columns:", pf.metadata.num_columns)

# ===== 2. schema =====
print("\n=== Schema / 列的数据类型 ===")
schema = pf.schema_arrow
for field in schema:
    print(f"{field.name}  -->  {field.type}")

# ===== 3. row group 详情 =====
print("\n=== Row Group Details ===")
for i in range(pf.num_row_groups):
    rg = pf.metadata.row_group(i)
    print(f"Row group {i}: rows={rg.num_rows}, total_byte_size={rg.total_byte_size}")

# ===== 4. 读取样本 =====
dataset = ds.dataset(file_path, format="parquet")
sample_n = 10
sample_table = dataset.head(sample_n)
sample_df = sample_table.to_pandas()

print(f"\n=== 前{sample_n}条样本 Sample Rows ===")
display(sample_df.head(15))

# ===== 5. pandas dtype =====
print("\n=== Pandas dtypes ===")
print(sample_df.dtypes)

# ===== 6. Python 值类型 =====
print("\n=== 每列样本值的 Python 类型 ===")
for col in sample_df.columns:
    first_valid = None
    for v in sample_df[col]:
        if pd.notna(v):
            first_valid = v
            break
    print(f"{col}: {type(first_valid).__name__ if first_valid is not None else 'All null'}")

# ===== 7. 缺失值统计 =====
print("\n=== Null Counts / Ratios (sample-based) ===")
for col in sample_df.columns:
    null_count = sample_df[col].isna().sum()
    null_ratio = null_count / len(sample_df)
    print(f"{col}: null_count={null_count}, null_ratio={null_ratio:.2%}")

# ===== 8. 数值列统计 =====
num_df = sample_df.select_dtypes(include=["number"])
if not num_df.empty:
    print("\n=== Numeric Column Statistics (sample-based) ===")
    print(num_df.describe().T)

# ===== 9. 字符串长度统计 =====
print("\n=== String Length Statistics (sample-based) ===")
for col in sample_df.select_dtypes(include=["object", "string"]).columns:
    lengths = sample_df[col].dropna().astype(str).map(len)
    if len(lengths) > 0:
        print(
            f"{col}: count={len(lengths)}, mean={lengths.mean():.2f}, "
            f"median={lengths.median():.2f}, min={lengths.min()}, max={lengths.max()}"
        )

# ===== 10. 唯一值数量 =====
print("\n=== Unique Value Counts (sample-based) ===")
for col in sample_df.columns:
    print(f"{col}: unique={sample_df[col].nunique(dropna=True)}")

# ===== 11. FineWeb 常见字段专项统计 =====
if "token_count" in sample_df.columns:
    print("\n=== token_count stats ===")
    print(sample_df["token_count"].describe())

if "language_score" in sample_df.columns:
    print("\n=== language_score stats ===")
    print(sample_df["language_score"].describe())

if "text" in sample_df.columns:
    text_lengths = sample_df["text"].dropna().astype(str).map(len)
    print("\n=== text length stats ===")
    print(text_lengths.describe())

if "url" in sample_df.columns:
    domains = sample_df["url"].dropna().astype(str).map(
        lambda x: urlparse(x).netloc.lower().removeprefix("www.")
    )
    print("\n=== Top Domains (sample-based) ===")
    print(domains.value_counts().head(20))

# ===== 12. 导出样本 =====
# output_file = "fineweb_sample.csv"
# sample_df.to_csv(output_file, index=False)
# print(f"\n已导出到 csv 文件: {os.path.abspath(output_file)}")

# ===== 13. batch 读取预览 =====
print("\n=== Batch Reading Preview ===")
scanner = dataset.scanner(batch_size=1000)
for i, batch in enumerate(scanner.to_batches()):
    batch_df = batch.to_pandas()
    print(f"Batch {i}: rows={len(batch_df)}")
    print(batch_df.head(3))
    if i >= 1:
        break

=== 基本信息 Basic Info ===
File path: /home/jovyan/data/fineweb/004_00048.parquet
File size: 339.15 MB
Row groups: 154
Number of rows: 153387
Number of columns: 9

=== Schema / 列的数据类型 ===
text  -->  string
id  -->  string
dump  -->  string
url  -->  string
date  -->  string
file_path  -->  string
language  -->  string
language_score  -->  double
token_count  -->  int64

=== Row Group Details ===
Row group 0: rows=1000, total_byte_size=3414246
Row group 1: rows=1000, total_byte_size=3943977
Row group 2: rows=1000, total_byte_size=3711911
Row group 3: rows=1000, total_byte_size=4201493
Row group 4: rows=1000, total_byte_size=3754738
Row group 5: rows=1000, total_byte_size=3836305
Row group 6: rows=1000, total_byte_size=4031725
Row group 7: rows=1000, total_byte_size=3922315
Row group 8: rows=1000, total_byte_size=3838461
Row group 9: rows=1000, total_byte_size=3640501
Row group 10: rows=1000, total_byte_size=3989095
Row group 11: rows=1000, total_byte_size=3718404
Row group 12: rows=1000, t

,text,id,dump,url,date,file_path,language,language_score,token_count
0,The passion of The Captain was incredible. Now...,<urn:uuid:309f1d36-0c35-4272-8f69-6a9513078b2b>,CC-MAIN-2025-26,https://on2victory.com/collections/artworks/pr...,2025-06-14T17:40:50Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.926981,164
1,We believe that all people are created equally...,<urn:uuid:2ce59b90-e4dc-432b-bafb-9aa8361d93b9>,CC-MAIN-2025-26,https://oneillcareerhub.indiana.edu/resources/...,2025-06-14T17:11:25Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.957867,83
2,Walleye Fishing on the Niagara River\nA sure s...,<urn:uuid:15237ff0-717d-4b80-a314-e8336cb9fdd6>,CC-MAIN-2025-26,https://onemoredrift.com/walleye/,2025-06-14T18:04:20Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.938820,199
3,Watch your favourite content come to life with...,<urn:uuid:d9ec0d7b-e542-4fa3-b504-cebfaf0b2774>,CC-MAIN-2025-26,https://onestopshopall.com/product/lg-138-cm-5...,2025-06-14T17:38:32Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.867190,430
4,If you want to convert AZW files to CBR format...,<urn:uuid:cd1ff226-f65b-45c6-93b7-c80c5529e20e>,CC-MAIN-2025-26,https://online-converters.com/convert/document...,2025-06-14T17:57:44Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.943740,428
5,Converting BIN to RTF files can be a convenien...,<urn:uuid:b9a12c09-8c24-4148-848d-78ba07d1c987>,CC-MAIN-2025-26,https://online-converters.com/convert/document...,2025-06-14T17:09:26Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.942688,566
6,It is important for a vehicle owner to look fo...,<urn:uuid:60f83f40-cdcf-453f-9272-77447c300936>,CC-MAIN-2025-26,https://onlineinformationworld.com/signs-that-...,2025-06-14T17:42:23Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.950486,319
7,Bitcoin (BTC) shines as the undeniable pioneer...,<urn:uuid:02fa6294-3e6e-497d-8775-65dbfa30ed88>,CC-MAIN-2025-26,https://onlycrafting.com/analysis-of-the-bitco...,2025-06-14T17:23:54Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.929800,630
8,"If you’re a Windows Insider on the Fast Ring, ...",<urn:uuid:418988a2-50b3-4cbc-9c7e-22305a4addd9>,CC-MAIN-2025-26,https://onmsft.com/news/msn-weather-is-finally...,2025-06-14T16:59:24Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.943082,200
9,NBC's latest channel is on YouTube\nOne the ma...,<urn:uuid:288294d8-aed5-4be8-8daa-001e5bb10c46>,CC-MAIN-2025-26,https://open.typepad.com/open/2006/06/nbcs_lat...,2025-06-14T16:20:38Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.949918,155



=== Pandas dtypes ===
text                  str
id                    str
dump                  str
url                   str
date                  str
file_path             str
language              str
language_score    float64
token_count         int64
dtype: object

=== 每列样本值的 Python 类型 ===
text: str
id: str
dump: str
url: str
date: str
file_path: str
language: str
language_score: float
token_count: int

=== Null Counts / Ratios (sample-based) ===
text: null_count=0, null_ratio=0.00%
id: null_count=0, null_ratio=0.00%
dump: null_count=0, null_ratio=0.00%
url: null_count=0, null_ratio=0.00%
date: null_count=0, null_ratio=0.00%
file_path: null_count=0, null_ratio=0.00%
language: null_count=0, null_ratio=0.00%
language_score: null_count=0, null_ratio=0.00%
token_count: null_count=0, null_ratio=0.00%

=== Numeric Column Statistics (sample-based) ===
                count        mean         std       min         25%  \
language_score   10.0    0.935057    0.025583   0.86719    0.93205

In [2]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 46.5 MB/s  0:00:01 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
